# Phase II, Pair 1: question D — front days inside the model

*What differs inside the model between a normal day and a front day, and can the state predict that the forecast will fail?*

| | |
|---|---|
| Authors | |
| Status | in progress |
| Phase I result this builds on | Sandro's front runs (30 front starts against 306 normal starts: about 5.3 °C too warm 3 h after the front, band holds 32 % against 81 % in normal weather) and Exp 11 (band honest on normal days, half the hours below the band with a front coming) |

## 1 · Question

What differs inside the model between a normal day and a front day, and can the state predict that the forecast will fail?

Two words need a fixed meaning for the whole notebook:

- **Front day** (the "special" case): a forecast start 12 h or 6 h before the peak of a cold front in `fronts.csv`
  (`cold_front == True`), i.e. the `front-12` and `front-6` starts of Phase I. Normal starts whose forecast window or last
  day of context contains a `cold_front` passage at that point in `events.parquet` are neither normal nor front: they are
  kept out of the normal reference.
- **Fails**: the mean absolute error of the median over the 24 forecast hours is above **2 K**. (Phase I counted hours with
  an error above 2 K; the band-miss definition is not used for "fails", band coverage is reported on its own.)

## 2 · Why this question

Phase I showed *that* the forecast breaks at fronts: the error rises to 5–7 °C and the 10–90 % band holds only 32 % of
the hours, against 81 % in normal weather. The band does not widen before a front, so the output does not show that the
model is unsure. We do not know whether anything *inside* the model is different on those days. If the state before a
front already sits apart from the states of normal days, the model "has" the information but does not turn it into a
wider band. If it does not, the failure is invisible to the model.

## 3 · Hypothesis and what would count as an answer

**Hypothesis.** Front states sit apart from normal states, and the distance from the normal states goes with failure.

- **If the hypothesis is right, we would see:** front states farther from their nearest normal states than normal states
  are from each other (also against temperature-matched normal cases), a positive rank correlation between that distance
  and the error, a higher fail rate and lower band coverage in the far distance bins, and a distance that separates
  failures from successes (AUC clearly above 0.5) when splitting by point and by month.
- **If it is wrong, we would see:** front and normal states mixed (same distance distribution), no relation between
  distance and error, and coverage that does not depend on the distance: the failures are invisible in the state.

## 4 · Setup

*Imports, paths, model loading.*

### The "(check)" points (source: `tirex-2==0.3.1` as pinned, and `src/tirex2/model/tirex2.py` on `main`)

- **`input_patch_size`**: set by the checkpoint's config, printed below. Phase I Exp 10 found the important hours repeating
  every 32 h, which points to patches of 32 h.
- **Number of blocks**: `len(core.stack)`, printed below; one `MultivariateBlock` (time mixer, then variate attention)
  per entry of the config's `stack_config.recipe`.
- **`_forward_layerwise`**: `_forward_layerwise(self, x, group_vector, target_mask, known_covariate_mask)` in 0.3.1;
  `main` adds `indices=None`. It is a generator that yields the tokens after each block, `[rows, tokens, dim]`. We do not
  call it ourselves: forward hooks on `core.stack[b]` see the same tensors.
- **Grad enabled**: the forward pass itself has no `no_grad`, but `TiRex2.predict` (and so `model.forecast`) is decorated
  with `@torch.no_grad`. `_predict(..., preserve_grad=True)` keeps the graph. D and E do not need gradients.

### The state

**State = output of the last block at the last context patch** (the token holding the newest observed hours), for the row
of the point. The input is NaN-padded on the right by `future_len`, so the last tokens are the forecast window and the
last *context* token is `L - future_len // input_patch_size - 1`. With sign-flip TTA, `predict` runs twice; the state is
taken from the first (unflipped) pass.

The function lives in `model_state.py` next to the notebooks. **E imports the same function**, so both questions use
exactly the same state.

In [ ]:
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from model_state import context_token, hidden_states, state_and_forecast
from tirex2 import load_model

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
DATA = Path(os.environ.get("AIVIS_DATA", ROOT / "data" / "austria_15km"))
CKPT = os.environ.get("TIREX_CKPT", "NX-AI/TiRex-2")
DEVICE = os.environ.get("TIREX_DEVICE", "cpu")
RESULTS = ROOT / "data" / "notebooks" / "results" / "04_question_d"
RESULTS.mkdir(parents=True, exist_ok=True)

CONTEXT, HORIZON, SEED = 672, 24, 0

model = load_model(CKPT, device=DEVICE)  # as in phase1-model-behavior/pair-1/run_forecasts.py
core = model.model
h = hidden_states(model, np.zeros((1, CONTEXT), np.float32))
print(f"input_patch_size {core.input_patch_size} | blocks {len(core.stack)} | embedding_dim {core.embedding_dim}")
print(
    f"tokens per input {h.shape[2]} | state token {context_token(core, h.shape[2])} | context_len {core.context_len}"
    f" | future_len {core.future_len} | tta_sign_flip {core.tta_sign_flip}"
)

## 5 · Data

Phase I design: 672 h context, 24 h horizon, start times and points of `tirex2_forecasts.npz` (336 starts × 378 points:
306 normal grid starts and 30 front starts). The context is filled exactly as in `run_forecasts.py`.

`FULL = True` uses all 336 × 378 cases (about 75 min on CPU, as in Phase I). The default is a sample:
**all front starts**, plus for each front start `N_MATCH` normal starts at the nearest hour of day and the nearest date
(at least 3 days away from any cold-front peak), at `N_POINTS` points spread over Austria (farthest-point sampling).
Matching by season and hour of day keeps the daily cycle and the season from separating the groups on their own.

The states (and the forecasts computed with them) are saved to `data/notebooks/results/04_question_d/`, not to git.

In [ ]:
FULL, N_MATCH, N_POINTS = False, 2, 120
FAIL_K = 2.0  # "fails": MAE of the median over 24 h above this (K)

pts = pd.read_csv(DATA / "points_with_labels.csv").set_index("point_id")
grid = pd.read_parquet(DATA / "grid_long.parquet", columns=["point_id", "time", "T2M"])
OBS = grid.pivot(index="time", columns="point_id", values="T2M")
T = OBS.interpolate(limit_area="inside").ffill().bfill()  # as run_forecasts.py
times, pids = T.index, T.columns.to_numpy()
SER = T.to_numpy(np.float32)
OBSV = OBS.to_numpy(np.float32)
fronts = pd.read_csv(DATA / "fronts.csv", parse_dates=["peak_time"])
front_peaks = times.get_indexer(pd.to_datetime(fronts[fronts.cold_front].peak_time, utc=True))

P1 = np.load(DATA / "tirex2_forecasts.npz", allow_pickle=True)
tags = P1["tags"].astype(str)
origin = P1["origin_idx"].astype(int)
p1_col = {p: k for k, p in enumerate(P1["pids"])}
assert (P1["times"] == times.astype("int64").to_numpy()).all(), "time axis differs from the Phase I file"

front_k = np.where(np.char.startswith(tags, "front"))[0]
grid_k = np.where(tags == "grid")[0]
if FULL:
    start_k = np.r_[grid_k, front_k]
else:
    far = grid_k[[np.abs(front_peaks - origin[k]).min() >= 72 for k in grid_k]]
    hour = lambda k: times[origin[k]].hour
    match = set()
    for k in front_k:
        cand = sorted(
            far, key=lambda g: (min(abs(hour(g) - hour(k)), 24 - abs(hour(g) - hour(k))), abs(origin[g] - origin[k]))
        )
        match.update(int(g) for g in cand[:N_MATCH])
    start_k = np.r_[np.sort(list(match)).astype(int), front_k]

XY = pts.loc[pids, ["x", "y"]].to_numpy(float) / 1000
if FULL or N_POINTS >= len(pids):
    chosen = np.arange(len(pids))
else:
    rng = np.random.default_rng(SEED)
    chosen = [int(rng.integers(len(pids)))]
    while len(chosen) < N_POINTS:
        d = np.linalg.norm(XY[:, None] - XY[chosen][None], axis=-1).min(axis=1)
        chosen.append(int(np.argmax(d)))
    chosen = np.array(chosen)

cases = pd.DataFrame(
    [(int(k), origin[k], tags[k], int(j), pids[j]) for k in start_k for j in chosen],
    columns=["p1_k", "i", "tag", "j", "pid"],
)
print(f"{len(start_k)} starts ({np.isin(start_k, front_k).sum()} front) x {len(chosen)} points = {len(cases)} cases")

In [ ]:
path = RESULTS / "states.npz"
key = cases[["i", "pid"]].to_numpy()
if path.exists() and np.array_equal(np.load(path)["key"], key):
    z = np.load(path)
    STATES, FRESH = z["states"], z["preds"]
else:
    t0 = time.time()
    ctx = np.stack([SER[i - CONTEXT : i, j] for i, j in zip(cases.i, cases.j)])
    STATES, FRESH = state_and_forecast(model, ctx, HORIZON, batch_size=128)
    np.savez_compressed(path, key=key, states=STATES, preds=FRESH)
    print(f"states for {len(cases)} cases in {time.time() - t0:.0f}s")
print("states", STATES.shape)

### Labels

- **Group**: `front` (front-12 / front-6 starts), `normal` (grid starts with no `cold_front` passage at that point from 24 h
  before to 24 h after the start in `events.parquet`), `excluded` (grid starts with such a passage).
- **Error, fail and coverage** from the Phase I forecasts in `tirex2_forecasts.npz`, so the model is not run again for
  them: MAE of the median over 24 h against the observed temperature, `fail = MAE > 2 K`, and the share of the 24 hours
  inside the 10–90 % band.
- **Control variables** for the checks: temperature of the last 24 h of context, month, hour of day.

The forecasts computed together with the states must reproduce the Phase I file (checked here for every case).

In [ ]:
ev = pd.read_parquet(DATA / "events.parquet", columns=["point_id", "time", "kind"])
ev = ev[ev.kind == "cold_front"]
ev_t = times.get_indexer(pd.to_datetime(ev.time, utc=True))
passage = {}
for p, t in zip(ev.point_id, ev_t):
    passage.setdefault(p, []).append(t)


def has_passage(pid, i):
    return any(i - 24 <= t < i + HORIZON for t in passage.get(pid, []))


cases["group"] = np.where(
    cases.tag.str.startswith("front"),
    "front",
    np.where([has_passage(p, i) for p, i in zip(cases.pid, cases.i)], "excluded", "normal"),
)
PRED = np.stack([P1["preds"][k, p1_col[p]] for k, p in zip(cases.p1_k, cases.pid)]).astype(np.float32)
obs = np.stack([OBSV[i : i + HORIZON, j] for i, j in zip(cases.i, cases.j)])
cases["mae"] = np.nanmean(np.abs(PRED[:, 4] - obs), axis=1)
cases["fail"] = cases.mae > FAIL_K
inside = ((obs >= PRED[:, 0]) & (obs <= PRED[:, 8])).astype(float)
cases["coverage"] = np.nanmean(np.where(np.isnan(obs), np.nan, inside), axis=1)
cases["t_last"] = [SER[i - 24 : i, j].mean() for i, j in zip(cases.i, cases.j)]
cases["month"] = times[cases.i].month
cases["hour"] = times[cases.i].hour

diff = np.abs(FRESH - PRED)
print(f"fresh forecast vs tirex2_forecasts.npz: max |diff| {diff.max():.4f} °C, mean {diff.mean():.5f} °C")
cases.groupby("group").agg(
    n=("mae", "size"),
    mae=("mae", "mean"),
    fail_rate=("fail", "mean"),
    coverage=("coverage", "mean"),
    t_last=("t_last", "mean"),
).round(3)

## 6 · Technique 1 — 2D projection

States standardised per dimension, then PCA (50 components), then UMAP (`random_state` fixed). Coloured by group and by
error.

*What it shows:* whether front and normal states form separate regions, and whether high errors gather somewhere.
*What it cannot show:* distances. UMAP keeps neighbourhoods, not distances, and can draw clusters that are not there, so it
is an illustration only; the evidence is the distance in the full state space (Technique 2).

In [ ]:
import umap
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

Z = StandardScaler().fit_transform(STATES)
pca = PCA(n_components=min(50, Z.shape[1], len(Z) - 1), random_state=SEED).fit(Z)
ZP = pca.transform(Z)
print(
    f"PCA: first 2 components {pca.explained_variance_ratio_[:2].sum():.0%}, 50 components {pca.explained_variance_ratio_.sum():.0%}"
)
EMB = umap.UMAP(n_neighbors=30, min_dist=0.1, random_state=SEED).fit_transform(ZP)
cases["u1"], cases["u2"] = EMB[:, 0], EMB[:, 1]
cases["pc1"], cases["pc2"] = ZP[:, 0], ZP[:, 1]

colors = {"normal": "tab:gray", "excluded": "tab:olive", "front": "tab:red"}
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for row, (a, b, name) in enumerate([("pc1", "pc2", "PCA"), ("u1", "u2", "UMAP")]):
    ax = axes[row, 0]
    for g in ["normal", "excluded", "front"]:
        d = cases[cases.group == g]
        ax.scatter(d[a], d[b], s=4 if g != "front" else 8, c=colors[g], label=f"{g} ({len(d)})", alpha=0.6)
    ax.set(title=f"{name}: group", xticks=[], yticks=[])
    ax.legend(markerscale=3, fontsize=8)
    sc = axes[row, 1].scatter(cases[a], cases[b], s=4, c=cases.mae.clip(upper=6), cmap="viridis")
    axes[row, 1].set(title=f"{name}: MAE of the 24 h forecast (°C, clipped at 6)", xticks=[], yticks=[])
    fig.colorbar(sc, ax=axes[row, 1])
fig.savefig(RESULTS / "projection_static.png", dpi=150, bbox_inches="tight")

## 7 · Technique 2 — distance to the nearest normal states

For every case: mean Euclidean distance (standardised full state) to its `K` nearest **normal** states. Neighbours at the
**same point less than 28 days apart are left out**, the case itself included: their 672 h contexts overlap, so they
would be close for a trivial reason.

Then: is the distance larger for front cases? Is it also larger than for *normal cold* cases (each front case paired with
the normal case at the same point whose last-24 h temperature is closest), so that it is not just the cold? How does the
distance relate to the error (Spearman, bootstrap over start times), the fail rate and the band coverage per distance bin?

*What it shows:* whether the model's internal state "knows" the case is unusual, measured where it matters. *What it
cannot show:* why; and a distance is only relative to the normal cases we have.

In [ ]:
from IPython.display import display
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score

K, SAME_POINT_DAYS = 10, 28


def knn_distance(query, ref, q_meta, r_meta, k=K, chunk=512):
    """Mean distance to the k nearest reference rows, leaving out the same point within SAME_POINT_DAYS."""
    out = np.empty(len(query))
    rn = (ref**2).sum(1)
    for s in range(0, len(query), chunk):
        q = query[s : s + chunk]
        d2 = (q**2).sum(1)[:, None] + rn[None] - 2 * q @ ref.T
        same = (q_meta[s : s + chunk, None, 0] == r_meta[None, :, 0]) & (
            np.abs(q_meta[s : s + chunk, None, 1] - r_meta[None, :, 1]) < SAME_POINT_DAYS * 24
        )
        d2[same] = np.inf
        out[s : s + chunk] = np.sqrt(np.maximum(np.partition(d2, k, axis=1)[:, :k], 0)).mean(1)
    return out


meta = cases[["pid", "i"]].to_numpy()
normal = (cases.group == "normal").to_numpy()
cases["dist"] = knn_distance(Z, Z[normal], meta, meta[normal])

# temperature-matched normal cold cases: per front case, the normal case at the same point with the closest t_last
cold_pair = []
for _, c in cases[cases.group == "front"].iterrows():
    d = cases[normal & (cases.pid == c.pid).to_numpy()]
    if d.empty:
        continue
    cold_pair.append(int((d.t_last - c.t_last).abs().idxmin()))
cases["cold_match"] = False
cases.loc[cold_pair, "cold_match"] = True


def boot(f, df, n=1000, seed=SEED):
    r = np.random.default_rng(seed)
    starts = df.i.unique()
    vals = [f(df[df.i.isin(r.choice(starts, len(starts)))]) for _ in range(n)]
    return np.nanpercentile(vals, [2.5, 97.5])


rho = lambda d: spearmanr(d.dist, d.mae).statistic
rows = []
for name, d in [
    ("all", cases[cases.group != "excluded"]),
    ("normal", cases[normal]),
    ("front", cases[cases.group == "front"]),
]:
    lo, hi = boot(rho, d)
    auc = roc_auc_score(d.fail, d.dist) if d.fail.nunique() == 2 else np.nan
    rows.append(
        {
            "cases": name,
            "n": len(d),
            "dist_median": d.dist.median(),
            "spearman_dist_mae": rho(d),
            "ci_lo": lo,
            "ci_hi": hi,
            "auc_dist_for_fail": auc,
        }
    )
rows.append(
    {
        "cases": "normal, temperature-matched to fronts",
        "n": len(cold_pair),
        "dist_median": cases.loc[cold_pair, "dist"].median(),
    }
)
dist_table = pd.DataFrame(rows)
dist_table.to_csv(RESULTS / "distance_table.csv", index=False)

used = cases[cases.group != "excluded"].copy()
used["bin"] = pd.qcut(used.dist, 5, labels=[f"Q{q}" for q in range(1, 6)])
by_bin = used.groupby(["bin", "group"], observed=True).agg(
    n=("mae", "size"),
    dist=("dist", "median"),
    mae=("mae", "mean"),
    fail_rate=("fail", "mean"),
    coverage=("coverage", "mean"),
)
display(dist_table.round(3))
by_bin.round(3)

### Figures

In [ ]:
import plotly.graph_objects as go

# projection with a colour toggle (group / error / fail)
hover = [
    f"point {p}<br>{times[i]:%Y-%m-%d %H} UTC<br>{g}<br>MAE {m:.1f} °C"
    for p, i, g, m in zip(cases.pid, cases.i, cases.group, cases.mae)
]
group_colors = cases.group.map({"normal": "#9e9e9e", "excluded": "#b5b35c", "front": "#d62728"})
modes = {
    "group": {"color": group_colors, "showscale": False},
    "error (MAE, °C)": {"color": cases.mae.clip(upper=6), "colorscale": "Viridis", "showscale": True},
    "fail (MAE > 2 K)": {"color": np.where(cases.fail, "#d62728", "#9e9e9e"), "showscale": False},
}
fig = go.Figure(
    go.Scattergl(
        x=cases.u1, y=cases.u2, mode="markers", text=hover, hoverinfo="text", marker={"size": 4, **modes["group"]}
    )
)
fig.update_layout(
    title="UMAP of the model state (last block, last context patch) — illustration only",
    xaxis={"visible": False},
    yaxis={"visible": False},
    width=850,
    height=650,
    updatemenus=[
        {
            "buttons": [
                {"label": k, "method": "restyle", "args": [{f"marker.{a}": [v] for a, v in m.items()}]}
                for k, m in modes.items()
            ],
            "x": 0,
            "y": 1.08,
            "xanchor": "left",
        }
    ],
)
fig.write_html(RESULTS / "projection.html")
fig

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
ax = axes[0]
for g, c in [("normal", "tab:gray"), ("front", "tab:red")]:
    d = cases[cases.group == g]
    ax.scatter(d.dist, d.mae, s=5 if g == "normal" else 10, c=c, alpha=0.5, label=f"{g} ({len(d)})")
ax.axhline(2, color="k", lw=0.8, ls="--", label=f"fail threshold {FAIL_K:g} K")
ax.set(
    xlabel="distance to the 10 nearest normal states (standardised)",
    ylabel="MAE of the 24 h forecast (°C)",
    title="Distance to normal states against error",
)
ax.legend(markerscale=2, fontsize=8)

ax = axes[1]
cov = used.groupby(["bin", "group"], observed=True).coverage.mean().unstack()
x = np.arange(len(cov))
for n, (g, c) in enumerate([("normal", "tab:gray"), ("front", "tab:red")]):
    if g in cov:
        ax.bar(x + (n - 0.5) * 0.4, cov[g] * 100, 0.4, color=c, label=g)
ax.axhline(80, color="k", lw=0.8, ls="--", label="nominal 80 %")
ax.set_xticks(x, [f"{b}\n{used[used.bin == b].dist.median():.1f}" for b in cov.index])
ax.set(
    xlabel="distance bin (quintiles of all cases; median distance below)",
    ylabel="hours inside the 10–90 % band (%)",
    ylim=(0, 100),
    title="Band coverage per distance bin (Phase I: 81 % normal, 32 % at fronts)",
)
ax.legend(fontsize=8)
fig.savefig(RESULTS / "distance_error_coverage.png", dpi=150, bbox_inches="tight")

## 8 · Comparing the techniques

*Do they agree? Where do they differ?*

## 9 · Checks

*Does the result hold on other dates, other points, other groups? What did we try to break it with?*

## 10 · Figure decisions

| Figure | Keep / change / drop | Why |
|---|---|---|

## 11 · What it means

*Answer to the question, in plain words. What is still open?*

## 12 · What did not work

*Dead ends and why. They are part of the record.*